In [1]:
import pandas as pd
import numpy as np

In [2]:
data_path = "./candidates.parquet"

In [3]:
data = pd.read_parquet(data_path)
print(data.shape)
print(data.head(5))

(123108, 26)
  mission_id simulator_version policy  config_mission_duration  \
0  mission_1                v1   fifo                      100   
1  mission_1                v1   fifo                      100   
2  mission_1                v1   fifo                      100   
3  mission_1                v1   fifo                      100   
4  mission_1                v1   fifo                      100   

   config_storage_capacity  config_image_arrival_rate  \
0                     10.0                       10.0   
1                     10.0                       10.0   
2                     10.0                       10.0   
3                     10.0                       10.0   
4                     10.0                       10.0   

   config_const_take_image  config_contact_frequency  config_contact_duration  \
0                     True                         6                     10.0   
1                     True                         6                     10.0   
2   

In [4]:
#caculator greatest_loss - loss_eacH_candidate (where greatest loss is the greatest loss in each gruop)
dfs = {key : group for key, group in data.groupby("group_id")}
for key in dfs.keys():
    group_data = dfs[key]
    max_loss = group_data["rollout_loss"].max()
    group_data["rollout_loss"] = max_loss - group_data["rollout_loss"]

In [5]:
data = pd.concat(dfs).rename(columns = {"rollout_loss" : "value"})

In [182]:
# from sklearn.model_selection import GroupShuffleSplit
# rand_seed = 10

# data_split1 = GroupShuffleSplit(test_size=.30, n_splits=1, random_state = rand_seed).split(data, groups=data["group_id"])
# train_idx, temp_idx = next(data_split1)

# train_data = data.iloc[train_idx]

# x_train = train_data[["remaining_contact_days", "predicted_value", "size_mb"]]
# y_train = train_data["value"]

# temp_df = data.iloc[temp_idx]

# data_split2 = GroupShuffleSplit(test_size=.5, n_splits=1, random_state = rand_seed).split(temp_df, groups=temp_df["group_id"])
# eval_idx, test_idx = next(data_split2)

# eval_data = data.iloc[eval_idx]
# x_eval = eval_data[["remaining_contact_days", "predicted_value", "size_mb"]]
# y_eval = eval_data["value"]


# test_data = data.iloc[test_idx]
# x_test = test_data[["remaining_contact_days", "predicted_value", "size_mb"]]
# y_test = test_data["value"]

# import xgboost as xgb
# device = "cpu"

# model = xgb.XGBRanker(  
#     tree_method='hist',
#     device=device,
#     booster='gbtree',
#     objective='rank:pairwise',
#     random_state=rand_seed, 
#     learning_rate=0.1,
#     colsample_bytree=0.9, 
#     eta=0.05, 
#     max_depth=12, 
#     n_estimators=110, 
#     subsample=0.75 
#     )

# train_qid = train_data["group_id"]
# test_qid = test_data["group_id"]
# model.fit(x_train, y_train, qid=train_qid, 
#           eval_set = [(x_test, y_test.astype(int))],
#           eval_qid = [test_qid],
#           verbose=True)
# evals_result = model.evals_result()
# print(evals_result)

In [12]:
import sklearn
from sklearn.metrics import make_scorer
from sklearn.metrics import ndcg_score

sklearn.set_config(enable_metadata_routing=True)

def calculate_ndcg(y_true, y_pred, qid):
    raise NotImplementedError()

def calculate_ndcg(y_true, y_pred, qid):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    scores = []

    for group in np.unique(qid):
        mask = qid == group

        if mask.sum() < 2:
            continue

        score = ndcg_score(
            [y_true[mask]],
            [y_pred[mask]]
        )

        scores.append(score)

    return np.mean(scores) if scores else 0.0
    
ndcg_scorer = make_scorer(
    calculate_ndcg,
    greater_is_better=True,
).set_score_request(qid=True)

In [15]:
import xgboost as xgb
import tqdm
from sklearn.model_selection import GroupKFold, GridSearchCV

rand_seed = 10

n_outer_splits = 10
gkf_outer = GroupKFold(n_splits=n_outer_splits)

n_inner_splits = 10
gkf_inner = GroupKFold(n_splits=n_inner_splits)


features = ["remaining_contact_days", "predicted_value", "size_mb"]
outer_qid = data["group_id"]
p_grid = {
    "max_depth": [3, 8, 15],
    "learning_rate": [0.01, 0.05],
    "n_estimators": [100, 300],
    "colsample_bytree": [0.8, 1.0],
    "subsample": [0.75, 1.0],
    "reg_alpha": [0.1, 0.5],
    "reg_lambda": [2.5, 5.0],
}
device = "cpu"

In [20]:
outer_scores = {}
for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(gkf_outer.split(data[features], data["value"], groups=outer_qid)):
    x_outer_train = data.iloc[outer_train_idx][features]
    y_outer_train = data.iloc[outer_train_idx]["value"]
    
    x_outer_test = data.iloc[outer_test_idx][features]
    y_outer_test = data.iloc[outer_test_idx]["value"]


    search = GridSearchCV(
        estimator=xgb.XGBRanker(
            tree_method="hist",
            device=device,
            booster="gbtree",
            objective="rank:pairwise",
            random_state=rand_seed,
        ).set_fit_request(qid=True),
        param_grid=p_grid,
        cv=gkf_inner,
        scoring=ndcg_scorer,
        refit=True,
        n_jobs=1,
        verbose=2
    )
    
    search.fit(
        x_outer_train,
        y_outer_train,
        groups=outer_qid.iloc[outer_train_idx],  
        qid=outer_qid.iloc[outer_train_idx],     
    )
    
    best_params = search.best_params_
    best_inner_score = search.best_score_
    
    outer_predictions = search.best_estimator_.predict(x_outer_test)

    outer_ndcg = calculate_ndcg(
        y_true=y_outer_test,
        y_pred=outer_predictions,
        qid=qid_outer.iloc[outer_test_idx],
    )
    
    outer_scores[outer_fold] = {
        "score": outer_ndcg,
        "best_params": search.best_params_,
        "inner_score": search.best_score_,
    }

    print(f"Outer fold {outer_fold}: {outer_ndcg:.4f}")

Fitting 10 folds for each of 192 candidates, totalling 1920 fits
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, reg_alpha=0.1, reg_lambda=2.5, subsample=0.75; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, reg_alpha=0.1, reg_lambda=2.5, subsample=0.75; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, reg_alpha=0.1, reg_lambda=2.5, subsample=0.75; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, reg_alpha=0.1, reg_lambda=2.5, subsample=0.75; total time=   0.8s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, reg_alpha=0.1, reg_lambda=2.5, subsample=0.75; total time=   0.8s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, reg_alpha=0.1, reg_lambda=2.5, subsample=0.75; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.

KeyboardInterrupt: 